# Primitive Triangle Tester\nThis notebook tests each method of the `Triangle` primitive in separated blocks.

In [1]:
import torch
import conquer3d as c3d
import plotly.graph_objects as go
import numpy as np

# Generate 3 different types of triangles
acute_v0 = torch.tensor([0.0, 0.0, 0.0])
acute_v1 = torch.tensor([1.0, 0.0, 0.0])
acute_v2 = torch.tensor([0.5, 0.866, 0.0])

obtuse_v0 = torch.tensor([0.0, 2.0, 0.0])
obtuse_v1 = torch.tensor([2.0, 2.0, 0.0])
obtuse_v2 = torch.tensor([-1.0, 2.5, 0.0])

right_v0 = torch.tensor([0.0, 4.0, 0.0])
right_v1 = torch.tensor([1.0, 4.0, 0.0])
right_v2 = torch.tensor([0.0, 5.0, 0.0])

triangles = {
    "Acute": c3d.Triangle(acute_v0, acute_v1, acute_v2),
    "Obtuse": c3d.Triangle(obtuse_v0, obtuse_v1, obtuse_v2),
    "Right": c3d.Triangle(right_v0, right_v1, right_v2)
}

colors = {
    "Acute": "blue",
    "Obtuse": "red",
    "Right": "green"
}

def create_base_fig():
    fig = go.Figure()
    for name, tri in triangles.items():
        v0, v1, v2 = tri.v0.numpy(), tri.v1.numpy(), tri.v2.numpy()
        pts = np.vstack([v0, v1, v2, v0])
        fig.add_trace(go.Scatter3d(
            x=pts[:,0], y=pts[:,1], z=pts[:,2],
            mode='lines+markers',
            line=dict(color=colors[name]),
            marker=dict(color=colors[name]),
            name=f'{name} Triangle'
        ))
    return fig


## Basic Properties\nArea, Normals, Centroids, and AABBs.

In [2]:
print("--- Basic Properties ---")
for name, tri in triangles.items():
    print(f"[{name}] Area: {tri.compute_area():.4f}")
    print(f"[{name}] Normal: {tri.compute_normal().tolist()}")
    print(f"[{name}] Centroid: {tri.compute_centroid().tolist()}")
    aabb_min, aabb_max = tri.compute_aabb()
    print(f"[{name}] AABB: min={aabb_min.tolist()}, max={aabb_max.tolist()}\n")

fig = create_base_fig()
for name, tri in triangles.items():
    c = tri.compute_centroid().numpy()
    fig.add_trace(go.Scatter3d(
        x=[c[0]], y=[c[1]], z=[c[2]],
        mode='markers', marker=dict(size=5, color=colors[name], symbol='diamond'),
        name=f'{name} Centroid'
    ))
fig.update_layout(scene=dict(aspectmode='data'), title="Basic Properties (Centroid)")
fig.show()


--- Basic Properties ---
[Acute] Area: 0.4330
[Acute] Normal: [0.0, 0.0, 0.9999999403953552]
[Acute] Centroid: [0.5, 0.28866666555404663, 0.0]
[Acute] AABB: min=[0.0, 0.0, 0.0], max=[1.0, 0.8659999966621399, 0.0]

[Obtuse] Area: 0.5000
[Obtuse] Normal: [0.0, -0.0, 1.0]
[Obtuse] Centroid: [0.3333333432674408, 2.1666667461395264, 0.0]
[Obtuse] AABB: min=[-1.0, 2.0, 0.0], max=[2.0, 2.5, 0.0]

[Right] Area: 0.5000
[Right] Normal: [0.0, 0.0, 1.0]
[Right] Centroid: [0.3333333432674408, 4.333333492279053, 0.0]
[Right] AABB: min=[0.0, 4.0, 0.0], max=[1.0, 5.0, 0.0]



## Ray Intersection\nCasting rays against the triangles.

In [3]:
print("--- Ray Intersection ---")
fig = create_base_fig()

import random

for name, tri in triangles.items():
    # Setup ray
    normal = tri.compute_normal()
    centroid = tri.compute_centroid()
    
    # Generate random ray origin somewhere generally above the triangle
    offset = torch.tensor([random.uniform(-1, 1), random.uniform(-1, 1), random.uniform(1.5, 3.0)])
    ray_origin = centroid + offset * normal  # Use normal to ensure it's "above" in local space
    if normal[2] == 0:
        # Fallback if normal is somewhat degenerate or weird
        ray_origin = centroid + torch.tensor([random.uniform(-1, 1), random.uniform(-1, 1), random.uniform(1.5, 3.0)])

    # Generate random ray direction that roughly points toward the triangle
    # We aim at a random point in/around the triangle's bounds
    target = centroid + torch.tensor([random.uniform(-0.8, 0.8), random.uniform(-0.8, 0.8), 0.0])
    ray_dir = target - ray_origin
    ray_dir = ray_dir / torch.norm(ray_dir) # Normalize
    
    ray = c3d.Ray(ray_origin, ray_dir)
    
    hit, t, u, v = tri.is_intersect_ray(ray)
    print(f"[{name}] Ray hit? {hit} at t={t:.4f}")
    
    ro, rd = ray_origin.numpy(), ray_dir.numpy()
    # Draw ray line longer if it misses, or just past the hit point
    ray_end = ro + rd * (t * 1.5 if hit else 4.0)
    
    # Ray line
    fig.add_trace(go.Scatter3d(
        x=[ro[0], ray_end[0]], y=[ro[1], ray_end[1]], z=[ro[2], ray_end[2]],
        mode='lines', line=dict(color=colors[name], dash='dot'), name=f'{name} Ray'
    ))
    # Origin
    fig.add_trace(go.Scatter3d(
        x=[ro[0]], y=[ro[1]], z=[ro[2]],
        mode='markers', marker=dict(size=4, color=colors[name], symbol='circle-open'), name=f'{name} Ray Origin'
    ))
    
    if hit:
        hit_pt = ro + rd * t
        fig.add_trace(go.Scatter3d(
            x=[hit_pt[0]], y=[hit_pt[1]], z=[hit_pt[2]],
            mode='markers', marker=dict(size=6, color=colors[name], symbol='cross'), name=f'{name} Ray Hit'
        ))

fig.update_layout(scene=dict(aspectmode='data'), title="Ray Intersections (Randomized)")
fig.show()

--- Ray Intersection ---
[Acute] Ray hit? True at t=2.2189
[Obtuse] Ray hit? False at t=0.0000
[Right] Ray hit? False at t=0.0000


## Closest Point & Inside Testing\nProjecting spatial points onto the triangles.

In [4]:
print("--- Closest Point & Point Inside ---")
fig = create_base_fig()

for name, tri in triangles.items():
    # Pick a test point
    if name == "Acute": test_pt = torch.tensor([0.5, -1.0, 1.0]) + tri.v0
    elif name == "Obtuse": test_pt = torch.tensor([0.5, 1.0, 1.0]) + tri.v0
    else: test_pt = torch.tensor([0.5, 3.0, 1.0]) + tri.v0
    
    closest = tri.compute_closest_point(test_pt)
    inside_on_plane = tri.test_point_inside_on_tria_plane(test_pt)
    inside_strict = tri.test_point_inside(test_pt)
    
    print(f"[{name}] Closest to {test_pt.tolist()} is {closest.tolist()}")
    print(f"[{name}] Is projected inside? {inside_on_plane} | Is strictly inside? {inside_strict}\n")
    
    tp, cp = test_pt.numpy(), closest.numpy()
    fig.add_trace(go.Scatter3d(
        x=[tp[0], cp[0]], y=[tp[1], cp[1]], z=[tp[2], cp[2]],
        mode='lines+markers', marker=dict(size=4, color=colors[name], symbol='square'),
        line=dict(color=colors[name], dash='dash'), name=f'{name} Closest Pt'
    ))

fig.update_layout(scene=dict(aspectmode='data'), title="Closest Points")
fig.show()


--- Closest Point & Point Inside ---
[Acute] Closest to [0.5, -1.0, 1.0] is [0.5, 0.0, 0.0]
[Acute] Is projected inside? False | Is strictly inside? False

[Obtuse] Closest to [0.5, 3.0, 1.0] is [0.37837839126586914, 2.270270347595215, 0.0]
[Obtuse] Is projected inside? False | Is strictly inside? False

[Right] Closest to [0.5, 7.0, 1.0] is [0.0, 5.0, 0.0]
[Right] Is projected inside? False | Is strictly inside? False



## Circumcenters\nComparing standard circumcenter vs. strictly interior circumcenter (snaps to longest edge midpoint if outside).

In [5]:
print("--- Circumcenter ---")
fig = create_base_fig()

for name, tri in triangles.items():
    cc_raw = tri.compute_circumcenter(strict_inside=False)
    cc_strict = tri.compute_circumcenter(strict_inside=True)
    
    print(f"[{name}] Circumcenter (raw): {cc_raw.tolist()}")
    print(f"[{name}] Circumcenter (strict): {cc_strict.tolist()}\n")
    
    cc1, cc2 = cc_raw.numpy(), cc_strict.numpy()
    
    fig.add_trace(go.Scatter3d(
        x=[cc1[0]], y=[cc1[1]], z=[cc1[2]],
        mode='markers', marker=dict(size=5, color=colors[name], symbol='circle'),
        name=f'{name} CC (Raw)'
    ))
    
    if np.linalg.norm(cc1 - cc2) > 1e-5:
        fig.add_trace(go.Scatter3d(
            x=[cc2[0]], y=[cc2[1]], z=[cc2[2]],
            mode='markers', marker=dict(size=6, color=colors[name], symbol='cross'),
            name=f'{name} CC (Strict)'
        ))

fig.update_layout(scene=dict(aspectmode='data'), title="Circumcenters")
fig.show()


--- Circumcenter ---
[Acute] Circumcenter (raw): [0.5, 0.2886582016944885, 0.0]
[Acute] Circumcenter (strict): [0.5, 0.2886582016944885, 0.0]

[Obtuse] Circumcenter (raw): [1.0, 5.25, 0.0]
[Obtuse] Circumcenter (strict): [0.5, 2.25, 0.0]

[Right] Circumcenter (raw): [0.5, 4.5, 0.0]
[Right] Circumcenter (strict): [0.5, 4.5, 0.0]

